In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("student+performance.zip")

ValueError: Multiple files found in ZIP file. Only one file per ZIP: ['.student.zip_old', 'student.zip']

In [2]:
import pandas as pd
math = pd.read_csv("student-mat.csv",sep=";")
por = pd.read_csv("student-por.csv" , sep=";")
print(math.shape,por.shape)

ModuleNotFoundError: No module named 'pandas'

In [1]:
import pandas as pd

In [2]:
ath = pd.read_csv("student-mat.csv",sep=";")
por = pd.read_csv("student-por.csv" , sep=";")
print(math.shape,por.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'student-mat.csv'

In [3]:
import pandas as pd

math = pd.read_csv("student_data/student-mat.csv", sep=";")
por = pd.read_csv("student_data/student-por.csv", sep=";")

print(math.shape, por.shape)

(395, 33) (649, 33)


In [4]:
assert math["age"].between(15, 22).all()
assert math["G1"].between(0, 20).all()
assert math["G2"].between(0, 20).all()
assert math["G3"].between(0, 20).all()
print("All range checks passed")

All range checks passed


| Evidence | My result | Interpretation |
|---|---|---|
| Shape / row count | 395 rows, 33 columns | Matches the expected size of the Math file; one row is one student's record for one course. |
| Missing / duplicate check | 0 missing, 0 duplicates | No cleaning needed for gaps or repeated rows. |
| Key or total reconciliation | (still to fill in) | To be completed after the value counts. |

In [5]:
print(math["school"].value_counts(dropna=False))
print(math["internet"].value_counts(dropna=False))

school
GP    349
MS     46
Name: count, dtype: int64
internet
yes    329
no      66
Name: count, dtype: int64


In [6]:
por["school"].value_counts()

school
GP    423
MS    226
Name: count, dtype: int64

In [8]:
support = math.loc[
    (math["G3"] < 10) | (math["failures"] > 0) | (math["absences"] >= 10),
    ["school", "sex", "age", "studytime", "failures", "absences", "G1", "G2", "G3"]
].copy()

print(support.shape)

(195, 9)


In [9]:
support.sort_values(["G3", "absences"], ascending=[True, False]).head(10)


,school,sex,age,studytime,failures,absences,G1,G2,G3
128,GP,M,18,1,2,0,7,4,0
130,GP,F,15,3,2,0,12,0,0
131,GP,F,15,1,0,0,8,0,0
134,GP,M,15,2,0,0,9,0,0
135,GP,F,15,3,0,0,11,0,0
136,GP,M,17,2,0,0,10,0,0
137,GP,F,16,1,2,0,4,0,0
140,GP,M,15,4,0,0,7,9,0
144,GP,M,17,1,3,0,5,0,0
146,GP,F,15,2,3,0,6,7,0


In [1]:
support.sort_values(["G3", "absences"], ascending=[True, False]).head(10)

NameError: name 'support' is not defined

In [2]:
import pandas as pd

math = pd.read_csv("student_data/student-mat.csv", sep=";")
por = pd.read_csv("student_data/student-por.csv", sep=";")

support = math.loc[
    (math["G3"] < 10) | (math["failures"] > 0) | (math["absences"] >= 10),
    ["school", "sex", "age", "studytime", "failures", "absences", "G1", "G2", "G3"]
].copy()

print(math.shape, por.shape, support.shape)

(395, 33) (649, 33) (195, 9)


support.sort_values(["G3", "absences"], ascending=[True, False]).head(10)

In [4]:
support.sort_values(["G3", "absences"], ascending=[True, False]).head(10)


,school,sex,age,studytime,failures,absences,G1,G2,G3
128,GP,M,18,1,2,0,7,4,0
130,GP,F,15,3,2,0,12,0,0
131,GP,F,15,1,0,0,8,0,0
134,GP,M,15,2,0,0,9,0,0
135,GP,F,15,3,0,0,11,0,0
136,GP,M,17,2,0,0,10,0,0
137,GP,F,16,1,2,0,4,0,0
140,GP,M,15,4,0,0,7,9,0
144,GP,M,17,1,3,0,5,0,0
146,GP,F,15,2,3,0,6,7,0


In [5]:
(support["G3"] == 0).sum()

np.int64(38)

In [6]:
(support[support["G3"] == 0]["absences"] == 0).sum()


np.int64(38)

| Evidence | Your result | Interpretation |
|---|---|---|
| Shape / row count | 195 rows, 9 columns (from 395 students) | About 49% of Math students meet at least one support condition. |
| Missing / duplicate check | No missing values in the source columns | Filtering only selects rows, so it cannot create missing values. |
| Key or total reconciliation | 38 of 195 have G3 = 0, and all 38 have 0 absences | Zero grades likely mean no final exam or dropout, not a real score of 0.|

**Interpretation:** The support rule flags 195 of 395 Math students because a student needs only one of three conditions (G3 below 10, any failures, or 10+ absences). That is about half the cohort, which may be too broad for a school team to act on. 38 flagged students have a final grade of 0 and no absences, so those zeros may mean the exam was not taken, and this group should be checked separately.


In [9]:
m = math.copy()
m["passed"] = m["G3"].ge(10)

print(m.shape)
print(m["passed"].value_counts())

(395, 34)
passed
True     265
False    130
Name: count, dtype: int64


In [10]:
m["grade_change"] = m["G3"] - m["G1"]

print(m.shape)
print(m["grade_change"].describe())

(395, 35)
count    395.000000
mean      -0.493671
std        2.762482
min      -12.000000
25%       -1.000000
50%        0.000000
75%        1.000000
max        4.000000
Name: grade_change, dtype: float64


In [11]:
m["absence_band"] = pd.cut(
    m["absences"],
    [-1, 5, 10, 20, float("inf")],
    labels=["low", "moderate", "high", "very high"]
)

print(m.shape)
print(m["absence_band"].value_counts(dropna=False))

(395, 36)
absence_band
low          249
moderate      80
high          51
very high     15
Name: count, dtype: int64


In [12]:
text_cols = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob",
             "reason", "guardian", "schoolsup", "famsup", "paid", "activities",
             "nursery", "higher", "internet", "romantic"]

for c in text_cols:
    m[c] = m[c].astype("string").str.strip()

print(m.shape)
print(m[text_cols].equals(math[text_cols]))

(395, 36)
False


In [13]:
(m[text_cols].astype("object") == math[text_cols].astype("object")).all().all()

np.True_

| Evidence | Your result | Interpretation |
|---|---|---|
| Shape / row count | 395 rows; columns went from 33 to 36 | Three columns added (passed, grade_change, absence_band); no rows lost or duplicated. |
| Missing / duplicate check | Counts add up to 395 for passed (265 + 130) and absence_band (249 + 80 + 51 + 15) | No student fell outside a group. |
| Key or total reconciliation | Text columns identical to the original after stripping spaces | Cleaning changed no values; raw grades in math are untouched. |

**Interpretation:** I worked on a copy (m) so the raw data stayed unchanged, and added three columns: passed (G3 of 10 or more), grade_change (G3 minus G1) and absence_band. 265 students passed and 130 did not. The average grade change was about -0.49, but the minimum of -12 is probably a student with a final grade of 0, so those zeros pull the average down. Stripping spaces from the text columns changed nothing because they were already clean.

In [15]:
m.groupby(["school", "studytime"]).size()

school  studytime
GP      1             89
        2            176
        3             57
        4             27
MS      1             16
        2             22
        3              8
dtype: int64

In [16]:
summary = (
    m.groupby(["school", "studytime"], as_index=False)
    .agg(students=("G3", "size"),
         mean_final=("G3", "mean"),
         pass_rate=("passed", "mean"))
)

print(summary.shape)
summary

(7, 5)


,school,studytime,students,mean_final,pass_rate
0,GP,1,89,10.438202,0.696629
1,GP,2,176,10.119318,0.636364
2,GP,3,57,11.350877,0.736842
3,GP,4,27,11.259259,0.740741
4,MS,1,16,7.875000,0.375000
5,MS,2,22,10.590909,0.727273
6,MS,3,8,11.750000,0.875000


In [17]:
summary["pass_rate_pct"] = summary["pass_rate"] * 100

print(summary.shape)
summary.sort_values("mean_final", ascending=False)

(7, 6)


,school,studytime,students,mean_final,pass_rate,pass_rate_pct
6,MS,3,8,11.750000,0.875000,87.500000
2,GP,3,57,11.350877,0.736842,73.684211
3,GP,4,27,11.259259,0.740741,74.074074
5,MS,2,22,10.590909,0.727273,72.727273
0,GP,1,89,10.438202,0.696629,69.662921
1,GP,2,176,10.119318,0.636364,63.636364
4,MS,1,16,7.875000,0.375000,37.500000


In [18]:
assert summary["students"].sum() == len(m)
print(summary["students"].sum())

395


| Evidence | Your result | Interpretation |
|---|---|---|
| Shape / row count | summary has 7 rows and 6 columns | One row per school and study time combination; MS has no students with study time 4. |
| Missing / duplicate check | No missing values in the grouping columns | Every student landed in exactly one group. |
| Key or total reconciliation | students column sums to 395 (GP 349 + MS 46) | Grouping neither lost nor double-counted any student. |

**Interpretation:** Grouping by school and study time gave 7 groups, where one row is a group, not a student. In GP, students who study more (codes 3 and 4) have higher mean final grades (about 11.3) and pass rates (about 74%) than students with codes 1 and 2. The top group, MS with study time 3, has only 8 students, and the weakest, MS with study time 1, has 16, so small groups limit what we can conclude. This shows an association, not that studying more causes better grades. Final grades of 0 may also pull some group averages down.

In [20]:
keys = ["school", "sex", "age", "address", "famsize", "Pstatus", "Medu", "Fedu",
        "Mjob", "Fjob", "reason", "nursery", "internet"]

print("Math duplicate keys:", math.duplicated(keys).sum())
print("Portuguese duplicate keys:", por.duplicated(keys).sum())

Math duplicate keys: 4
Portuguese duplicate keys: 12


In [21]:
dups = math[math.duplicated(keys, keep=False)].sort_values(keys)
print(dups.shape)
dups[["school", "sex", "age", "address", "Medu", "Fedu", "G1", "G2", "G3"]]

(8, 33)


,school,sex,age,address,Medu,Fedu,G1,G2,G3
107,GP,M,16,U,3,3,16,18,18
181,GP,M,16,U,3,3,12,13,12
115,GP,M,16,U,4,4,15,15,16
148,GP,M,16,U,4,4,7,6,0
78,GP,M,17,U,2,1,8,8,10
144,GP,M,17,U,2,1,5,0,0
277,GP,M,18,U,4,4,9,9,9
341,GP,M,18,U,4,4,10,10,0


In [22]:
both = math.merge(
    por,
    on=keys,
    how="inner",
    suffixes=("_math", "_por"),
    validate="one_to_one",
    indicator=True
)

print(both.shape)

MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
 school sex  age address famsize Pstatus  Medu  Fedu     Mjob     Fjob reason nursery internet
    GP   M   17       U     GT3       T     2     1    other    other   home     yes      yes
    GP   M   16       U     GT3       T     4     4  teacher  teacher course     yes      yes
    GP   M   16       U     GT3       T     3     3 services    other   home     yes      yes
    GP   M   18       U     GT3       T     4     4  teacher services   home     yes      yes ...
Duplicates in right:
 school sex  age address famsize Pstatus  Medu  Fedu     Mjob     Fjob reason nursery internet
    GP   M   17       U     GT3       T     2     1    other    other   home     yes      yes
    GP   M   16       U     GT3       T     4     4  teacher  teacher course     yes      yes
    GP   M   16       U     GT3       T     3     3 services    other   home     yes      yes
    GP   M  

In [23]:
math_u = math[~math.duplicated(keys, keep=False)].copy()
por_u = por[~por.duplicated(keys, keep=False)].copy()

print("Math:", len(math), "->", len(math_u), "excluded:", len(math) - len(math_u))
print("Portuguese:", len(por), "->", len(por_u), "excluded:", len(por) - len(por_u))


Math: 395 -> 387 excluded: 8
Portuguese: 649 -> 626 excluded: 23


In [24]:
both = math_u.merge(
    por_u,
    on=keys,
    how="inner",
    suffixes=("_math", "_por"),
    validate="one_to_one",
    indicator=True
)

print(both.shape)
print(both["_merge"].value_counts())

(358, 54)
_merge
both          358
left_only       0
right_only      0
Name: count, dtype: int64


In [25]:
both["final_gap"] = both["G3_math"] - both["G3_por"]

print(both.shape)
print(both[["G3_math", "G3_por", "final_gap"]].describe())

(358, 55)
          G3_math      G3_por   final_gap
count  358.000000  358.000000  358.000000
mean    10.547486   12.583799   -2.036313
std      4.522181    2.948192    4.035739
min      0.000000    0.000000  -15.000000
25%      8.000000   11.000000   -4.000000
50%     11.000000   13.000000   -2.000000
75%     14.000000   14.750000    0.750000
max     20.000000   19.000000   11.000000


In [26]:
assert both.shape[0] <= min(len(math_u), len(por_u))
assert not both.duplicated(keys).any()
assert (both["_merge"] == "both").all()
print("All merge checks passed")

All merge checks passed


| Evidence | Your result | Interpretation |
|---|---|---|
| Shape / row count | Math 395 -> 387, Portuguese 649 -> 626 after exclusions; merged table 358 rows, 55 columns | 31 rows excluded because their key values were shared by lookalike students; 358 students matched in both files. |
| Missing / duplicate check | Duplicate keys: Math 4, Portuguese 12 before exclusion; 0 after | The first one_to_one merge failed, which proved the keys were not unique. After excluding lookalikes, it passed. |
| Key or total reconciliation | 358 is below the smaller table (387); no duplicate keys; every _merge value is "both" | No student was matched twice or lost during the merge. |

**Interpretation:** There is no student ID, so I matched students on 13 demographic and school columns, never on grades. The keys were not unique (4 duplicates in Math, 12 in Portuguese), and a one_to_one merge failed, which confirmed the problem. I excluded 8 Math and 23 Portuguese students who shared identical key values with someone else, because they cannot be matched reliably. That left 358 students in both files. On average these students scored about 2 points lower in Math than in Portuguese (10.55 vs 12.58). This is a fair comparison for the same students, but it does not show that students are worse at Math, because the two courses may be graded differently. Excluding the lookalike students may also make the matched group slightly unrepresentative.